# H-004 · Kalman β vs static OLS hedge (trad-z)

Requires frozen `BREAK_STAR` plus H-003 window STARs (`OLS_WINDOW_STAR`, `ADF_WINDOW_STAR`, `ENTRY_Z_STAR`, `Z_WINDOW_STAR`). Bake-off under that stack.

Freeze on fold-val Sharpe (max DD / corr to S1 as secondary). ADF / HL / lag-1 autocorr are **not** freeze inputs. Type `HEDGE_STAR` (`"ols"` or `"kalman"`); if Kalman, also type `KALMAN_DELTA_STAR`.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    fold_table,
    fold_val_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_universe_c_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    repo_root,
    split_is_oos,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.config import S2SimConfig

STAR_PATH = DEFAULT_STAR_STACK
TEARSHEET_DIR = ARTIFACTS_DIR
stack = load_star_stack(STAR_PATH)
PAIRS_STAR = list(stack["PAIRS_STAR"])
print("stack keys:", sorted(stack))
print("PAIRS_STAR", PAIRS_STAR)


## 1. Load frozen panel / PAIRS_STAR


In [ ]:
require_star("BAR_STAR", stack.get("BAR_STAR"))
for key in ("BREAK_STAR", "OLS_WINDOW_STAR", "ADF_WINDOW_STAR", "ENTRY_Z_STAR", "Z_WINDOW_STAR"):
    require_star(key, stack.get(key))
BAR = str(stack["BAR_STAR"])
train, full = load_universe_c_panels(BAR, PAIRS_STAR, root=ROOT)
lb = lookbacks_for_bar(
    BAR,
    ols_days=int(stack["OLS_WINDOW_STAR"]),
    z_days=int(stack["Z_WINDOW_STAR"]),
    adf_days=int(stack["ADF_WINDOW_STAR"]),
)
# Start from OLS panel with frozen windows; Kalman arm overlays in the bake-off.
train = overlay_ols_hedge(
    train,
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    adf_window=lb["adf_window"],
)
full = overlay_ols_hedge(
    full,
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    adf_window=lb["adf_window"],
)
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS rows", len(is_panel), "OOS rows", len(oos_panel))
print("windows", {k: lb[k] for k in ("ols_window", "z_window", "adf_window")})
is_panel.head()


## 2. Attach Kalman hedge columns (OLS panel already loaded)


In [ ]:
lb = lookbacks_for_bar(
    BAR,
    ols_days=int(stack["OLS_WINDOW_STAR"]),
    z_days=int(stack["Z_WINDOW_STAR"]),
    adf_days=int(stack["ADF_WINDOW_STAR"]),
)
KALMAN_DELTA_GRID = (1e-4, 1e-5, 1e-6, 1e-7)
CHAN_SANITY_DELTA = 1e-3


def _hedge_diag(panel, label):
    g = panel.groupby("pair_id")
    return pd.DataFrame(
        {
            "median_adf": g["adf_pvalue"].median(),
            "median_hl": g["half_life"].median(),
            "lag1_ac": g["spread"].apply(lambda s: float(s.astype(float).autocorr(lag=1))),
        }
    ).assign(arm=label)


kf_panels = {}
rows = [_hedge_diag(is_panel, "ols")]
for d in (CHAN_SANITY_DELTA,) + KALMAN_DELTA_GRID:
    p = overlay_kalman_hedge(
        is_panel.copy(),
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
        delta=d,
    )
    if d in KALMAN_DELTA_GRID:
        kf_panels[d] = p
    rows.append(_hedge_diag(p, f"kalman_{d:g}"))

diag = pd.concat(rows)
print("ADF/HL/autocorr diagnostic (not a freeze input). Expect lag1_ac and HL to rise toward OLS as δ shrinks.")
print("Chan δ=1e-4 is sanity contrast only — not a freeze candidate.")
diag


## 3. Walk-forward folds


In [ ]:
dates = pd.DatetimeIndex(pd.to_datetime(is_panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
fold_table(folds)


## 4. Fold-val metrics (validation only)


In [ ]:
cfg_ols = config_from_stack(stack, hedge="ols")
parts = [fold_val_metrics(is_panel, folds, {"ols": cfg_ols}, s1_weekly=s1_weekly)]
cfg_kf = config_from_stack(stack, hedge="kalman")
for d, panel in kf_panels.items():
    arm = f"kalman_{d:g}"
    parts.append(fold_val_metrics(panel, folds, {arm: cfg_kf}, s1_weekly=s1_weekly))
fold_df = pd.concat(parts, ignore_index=True)
fold_df


## 5. Boxplots (do not assign STAR here)


In [ ]:
plot_fold_boxplots(fold_df, title="H-004 OLS vs Kalman δ grid")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
fold_df.groupby("arm")[["ann_sharpe", "max_drawdown", "corr_to_s1"]].median()


## 6. Type `HEDGE_STAR` (and `KALMAN_DELTA_STAR` if Kalman) then save


In [ ]:
HEDGE_STAR = "ols"  # TODO after fold-val review: "ols" or "kalman"
KALMAN_DELTA_STAR = None  # TODO if Kalman: 1e-5, 1e-6, or 1e-7 (not 1e-4)
require_star("HEDGE_STAR", HEDGE_STAR)
stack["HEDGE_STAR"] = HEDGE_STAR
if HEDGE_STAR == "kalman":
    require_star("KALMAN_DELTA_STAR", KALMAN_DELTA_STAR)
    dstar = float(KALMAN_DELTA_STAR)
    if not any(abs(dstar - d) < 1e-12 for d in KALMAN_DELTA_GRID):
        raise ValueError(f"KALMAN_DELTA_STAR must be one of {KALMAN_DELTA_GRID}, got {KALMAN_DELTA_STAR!r}")
    stack["KALMAN_DELTA_STAR"] = dstar
else:
    stack["KALMAN_DELTA_STAR"] = None
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "HEDGE_STAR", HEDGE_STAR, "KALMAN_DELTA_STAR", stack.get("KALMAN_DELTA_STAR"))


## 7. Sealed OOS once


In [ ]:
require_star("HEDGE_STAR", HEDGE_STAR)
oos_use = oos_panel
if HEDGE_STAR == "kalman":
    require_star("KALMAN_DELTA_STAR", KALMAN_DELTA_STAR)
    oos_use = overlay_kalman_hedge(
        oos_panel.copy(),
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
        delta=float(KALMAN_DELTA_STAR),
    )
oos = run_s2_backtest(oos_use, config_from_stack(stack), s1_weekly=s1_weekly)
print(oos.metrics)
arm_label = HEDGE_STAR if HEDGE_STAR == "ols" else f"kalman_{float(KALMAN_DELTA_STAR):g}"
write_tearsheet_pdf(tearsheet_path("H-004", arm_label), oos.returns, title=f"H-004 {arm_label} sealed OOS")
print("tearsheet", tearsheet_path("H-004", arm_label))


## 8. Notes for next hyp


H-005 spread ADX/RSI filter is next. TA-Lib required for that notebook. Leave `HEDGE_STAR`, `KALMAN_DELTA_STAR`, `BREAK_STAR`, and window STARs frozen. If 1h is ever re-opened, Kalman Q should be ≈ Q_day/6 so calendar adaptation matches (do not session-scale δ on 1d).
